# Visualize crops, positive pairs, and augmentations

Used to tune `crop_size` and per-augmentation strengths before any training. Each cell is independent — re-run individual cells after editing `cfg`.


In [ ]:
# Cell 1: pick a zarr, build the dataset, print key facts, define visualization helpers.
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np

from config import Config
from data.picker import pick_zarr_path
from data.dataset import ZarrCellDataset

cfg = Config()
if cfg.zarr_path is None:
    cfg.zarr_path = pick_zarr_path()

ds = ZarrCellDataset(cfg)
sample_iter = iter(ds)

print('zarr_path           :', cfg.zarr_path)
print('arr.shape (T,C,Z,Y,X):', tuple(ds.arr.shape))
print('arr.dtype           :', ds.arr.dtype)
print('arr.chunks          :', ds.arr.chunks)
print('z_thickness         :', cfg.z_thickness)
print('delta_t frames      :', (ds.delta_t_min, ds.delta_t_max))
print('z range slices      :', (ds.z_min, ds.z_max))


def mean_projection(volume):
    """average z into one cleaner 2d view."""
    return volume.mean(axis=0)


def mip(volume):
    """max z into one bright 2d view."""
    return volume.max(axis=0)


def z_montage(volume, ncols=5):
    """tile z slices into one image."""
    z, h, w = volume.shape
    nrows = (z + ncols - 1) // ncols
    out = np.zeros((nrows * h, ncols * w), dtype=volume.dtype)
    for i in range(z):
        r, c = divmod(i, ncols)
        out[r*h:(r+1)*h, c*w:(c+1)*w] = volume[i]
    return out


def percentile_limits(values, low=0.5, high=99.0):
    """pick display limits from robust percentiles."""
    lo, hi = np.percentile(np.asarray(values), [low, high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.min(values)), float(np.max(values))
    if hi <= lo:
        hi = lo + 1e-6
    return float(lo), float(hi)


def raw_frame_volume(t, z0, z1):
    """load a full-frame z volume in raw intensity units."""
    frame = ds.arr[int(t), cfg.channel, int(z0):int(z1), :, :].compute()
    frame = np.asarray(frame, dtype=np.float32)
    return np.clip(frame, 0, cfg.max_intensity) / cfg.max_intensity


def raw_frame_mean(t, z0, z1):
    """load a full-frame mean projection in raw intensity units."""
    return raw_frame_volume(t, z0, z1).mean(axis=0)


def show_gray(ax, image, title, limits):
    """show a grayscale image with fixed limits."""
    ax.imshow(image, cmap='gray', vmin=limits[0], vmax=limits[1])
    ax.set_title(title)
    ax.axis('off')


In [ ]:
# Cell 2: pull one positive pair, show mean projections and their difference.
import matplotlib.pyplot as plt

pair = next(sample_iter)
v1 = pair['view1'][0].numpy()   # (z, h, w)
v2 = pair['view2'][0].numpy()
meta = pair['meta']
print('meta:', meta)
print('view shape (z, h, w):', v1.shape)

# these are normalized model inputs, so use one shared scale across the panels.
v1_mean = mean_projection(v1)
v2_mean = mean_projection(v2)
diff_mean = v1_mean - v2_mean
view_limits = percentile_limits(np.concatenate([v1_mean.ravel(), v2_mean.ravel()]), 1.0, 99.0)
diff_abs = np.percentile(np.abs(diff_mean), 99.0)
diff_limits = (-float(diff_abs), float(diff_abs)) if diff_abs > 0 else (-1.0, 1.0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
show_gray(axes[0], v1_mean, f"view1 mean z, t={int(meta['t1'])}", view_limits)
show_gray(axes[1], v2_mean, f"view2 mean z, t={int(meta['t2'])}", view_limits)
show_gray(axes[2], diff_mean, 'diff of mean projections', diff_limits)
plt.tight_layout(); plt.show()


In [ ]:
# Cell 2b: raw z montage and mean projection for the same un-augmented crop.
import matplotlib.pyplot as plt

m = meta
crop_h = crop_w = cfg.crop_size
raw = ds._load_crop(int(m['t1']), int(m['z']), int(m['y']), int(m['x']), crop_h, crop_w)
raw_volume = raw[0].numpy()  # (z, h, w)

z0 = int(m['z'])
z1 = z0 + cfg.z_thickness
full_volume = raw_frame_volume(int(m['t1']), z0, z1)
full_mean = full_volume.mean(axis=0)
raw_limits = percentile_limits(full_volume, 0.5, 99.0)
mean_limits = percentile_limits(full_mean, 0.5, 99.0)
print('full-volume display limits:', raw_limits)
print('full-mean display limits  :', mean_limits)
print('crop mean percentiles     :', np.percentile(mean_projection(raw_volume), [0.5, 50, 99.0]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show_gray(
    axes[0],
    z_montage(raw_volume),
    f'raw z montage, t={int(m["t1"])}, z=[{z0}, {z1})',
    raw_limits,
)
show_gray(axes[1], mean_projection(raw_volume), 'raw mean projection', mean_limits)
plt.tight_layout(); plt.show()


In [ ]:
# Cell 3: 4x4 grid of independently augmented variants of the same raw crop.
import matplotlib.pyplot as plt
from data.augmentations import build_augmentations

# reuse the same anchor so differences come from augmentation.
m = meta
crop_h = crop_w = cfg.crop_size
base = ds._load_crop(int(m['t1']), int(m['z']), int(m['y']), int(m['x']), crop_h, crop_w)

augment = build_augmentations(cfg)
aug_images = [mean_projection(augment(base.clone())[0].numpy()) for _ in range(16)]
aug_limits = percentile_limits(np.concatenate([img.ravel() for img in aug_images]), 1.0, 99.0)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img in zip(axes.flat, aug_images):
    show_gray(ax, img, '', aug_limits)
plt.suptitle('16 independent augmentations of the same crop, shown as mean projections')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 4: per-augmentation isolation shown as mean projections.
import copy
import matplotlib.pyplot as plt


def isolate_aug(name):
    # keep only one visual augmentation active.
    c = copy.deepcopy(cfg)
    for attr in ('aug_intensity', 'aug_gamma', 'aug_noise', 'aug_blur'):
        getattr(c, attr).enabled = (attr == name)
    c.aug_percrop_norm.enabled = True
    if name is not None:
        getattr(c, name).p = 1.0
    return build_augmentations(c)


def raw_aug():
    # keep only per-crop normalization.
    c = copy.deepcopy(cfg)
    for attr in ('aug_intensity', 'aug_gamma', 'aug_noise', 'aug_blur'):
        getattr(c, attr).enabled = False
    return build_augmentations(c)


modes = [
    ('raw norm only', raw_aug()),
    ('intensity only',  isolate_aug('aug_intensity')),
    ('gamma only',      isolate_aug('aug_gamma')),
    ('noise only',      isolate_aug('aug_noise')),
    ('blur only',       isolate_aug('aug_blur')),
    ('full stack',      build_augmentations(cfg)),
]

mode_images = [(title, mean_projection(aug(base.clone())[0].numpy())) for title, aug in modes]
mode_limits = percentile_limits(np.concatenate([img.ravel() for _, img in mode_images]), 1.0, 99.0)

fig, axes = plt.subplots(1, len(modes), figsize=(3 * len(modes), 3.5))
for ax, (title, img) in zip(axes, mode_images):
    show_gray(ax, img, title, mode_limits)
plt.suptitle('per-augmentation isolation, shown as mean projections')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 5: crop-size sweep on one anchor over a mean projection.
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# use full-frame percentile limits so empty regions stay dark.
z0 = int(meta['z'])
z1 = z0 + cfg.z_thickness
frame = raw_frame_mean(int(meta['t1']), z0, z1)
frame_limits = percentile_limits(frame, 0.5, 99.0)
print('frame display limits:', frame_limits)

x_off = int(meta['x']); y_off = int(meta['y'])
sizes = [64, 96, 128, 160, 224]
colors = ['tab:red', 'tab:orange', 'tab:green', 'tab:blue', 'tab:purple']

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(frame, cmap='gray', vmin=frame_limits[0], vmax=frame_limits[1])
for s, c in zip(sizes, colors):
    rect = patches.Rectangle(
        (x_off, y_off), s, s,
        linewidth=2, edgecolor=c, facecolor='none', label=f'crop_size={s}',
    )
    ax.add_patch(rect)
ax.legend(loc='upper right')
ax.set_title(f'crop-size sweep at t={int(meta["t1"])}, mean z=[{z0}, {z1})')
ax.axis('off')
plt.tight_layout(); plt.show()
